# MVP — Risco, Custo e Tempo de Implantação de uma Nova Unidade de Armazenamento em Nuvem

**Aluno(a):** _(preencha aqui)_
**Disciplina:** Sistemas de Suporte à Decisão

---

Este notebook constrói um **modelo de apoio à decisão (MVP)** para avaliar a implantação de uma nova unidade de armazenamento em nuvem (cloud storage), a partir da base pública do Kaggle **"Backblaze Hard Drive Stats"**, publicada pela **Backblaze** — empresa real de armazenamento em nuvem e backup que, desde 2013, disponibiliza dados operacionais de mais de 300 mil discos rígidos (HDD/SSD) em seus data centers.

O modelo combina três dimensões de decisão, uma por seção principal:

1. **Risco** — calculado a partir de dados reais de falha de disco (taxa de falha anualizada por modelo) e de um modelo preditivo (Random Forest) treinado sobre atributos S.M.A.R.T., que estima a probabilidade de falha de cada disco.
2. **Custo** — modelo heurístico com premissas de mercado (US$/TB por fabricante), combinando custo de aquisição e custo esperado de reposição por falhas.
3. **Tempo de implantação** — modelo heurístico que estima o prazo (em semanas) para colocar a nova unidade em operação, considerando aquisição, instalação e testes de queima (*burn-in*), ajustado pelo risco do modelo de disco escolhido.

Ao final, as três dimensões são combinadas em uma **matriz de decisão multicritério (MCDA)**, com análise de sensibilidade dos pesos e simulação de Monte Carlo para avaliar a robustez da recomendação.

> **Fonte dos dados:** [Backblaze Hard Drive Stats — Kaggle](https://www.kaggle.com/datasets/dansleboby/backblaze-hard-drive-stats) (dados também publicados oficialmente em [backblaze.com/cloud-storage/resources/hard-drive-test-data](https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data)). Baixe **um trimestre** do dataset (arquivo `.csv`, ex.: `2023-01-01.csv` a `2023-03-31.csv` de um mesmo trimestre) e faça upload conforme indicado na Seção 2.


## 1. Configuração do Ambiente

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Carregamento da Base de Dados

A base utilizada é o **Backblaze Hard Drive Stats**, com registros diários de cada disco em operação nos data centers da Backblaze: `date`, `serial_number`, `model`, `capacity_bytes`, `failure` (1 no dia em que o disco falhou, 0 nos demais dias) e dezenas de atributos **S.M.A.R.T.** (`smart_X_normalized` / `smart_X_raw`), que são indicadores de saúde reportados pelo próprio disco.

Baixe no Kaggle o(s) arquivo(s) `.csv` de **um trimestre** (ex.: os três meses de um mesmo trimestre) e faça upload abaixo. Se baixar mais de um arquivo, todos serão concatenados automaticamente.


In [ ]:
from google.colab import files
import io, os, glob

uploaded = files.upload()  # selecione um ou mais arquivos .csv do Backblaze Drive Stats

csv_files = [f for f in uploaded.keys() if f.lower().endswith('.csv')]
dfs = [pd.read_csv(f, low_memory=False) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
df.columns = [c.strip().lower() for c in df.columns]

print(f'Arquivos carregados: {csv_files}')
print(f'Base carregada: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head()


In [ ]:
# Verificação de qualidade dos dados
print('Período coberto:', df['date'].min(), 'a', df['date'].max())
print('Discos únicos (serial_number):', df['serial_number'].nunique())
print('Modelos únicos:', df['model'].nunique())
print('Total de falhas registradas:', int(df['failure'].sum()))
print('Linhas duplicadas:', df.duplicated().sum())
print()
print('% de valores ausentes (top 15 colunas com mais ausência):')
print((df.isna().mean().sort_values(ascending=False) * 100).head(15).round(1))


## 3. Preparação dos Dados

Nesta seção: (i) convertemos a capacidade para TB, (ii) inferimos o **fabricante** a partir do prefixo do nome do modelo (convenção real de nomenclatura da Backblaze) e (iii) selecionamos um subconjunto de atributos S.M.A.R.T. reconhecidos na literatura de confiabilidade de discos como bons preditores de falha (setores realocados, erros não corrigíveis, tempo ligado, entre outros).


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['capacity_tb'] = (df['capacity_bytes'] / (1024**4)).round(2)
df = df[df['capacity_tb'] > 0]  # remove linhas com capacidade inválida

def infer_manufacturer(model):
    m = str(model).upper()
    if m.startswith('ST'):
        return 'Seagate'
    if 'TOSHIBA' in m or m.startswith('MG') or m.startswith('MD') or m.startswith('MQ'):
        return 'Toshiba'
    if m.startswith('WDC') or m.startswith('WD') or m.startswith('WUH'):
        return 'Western Digital'
    if m.startswith('HGST') or m.startswith('HUH') or m.startswith('HUS') or m.startswith('HMS'):
        return 'HGST'
    return 'Outro'

df['fabricante'] = df['model'].apply(infer_manufacturer)

# Atributos S.M.A.R.T. classicamente associados a falha de disco (quando presentes na base)
smart_candidatos = [
    'smart_5_raw',    # setores realocados
    'smart_187_raw',  # erros reportados não corrigíveis
    'smart_188_raw',  # timeout de comando
    'smart_197_raw',  # setores pendentes atuais
    'smart_198_raw',  # setores não corrigíveis offline
    'smart_9_raw',    # horas ligado (idade do disco)
]
smart_cols = [c for c in smart_candidatos if c in df.columns]
if len(smart_cols) < 3:
    # fallback: pega colunas smart_*_raw numéricas disponíveis com baixa ausência
    todas_raw = [c for c in df.columns if c.startswith('smart_') and c.endswith('_raw')]
    ausencia = df[todas_raw].isna().mean().sort_values()
    smart_cols = list(ausencia.head(6).index)

print('Atributos S.M.A.R.T. selecionados:', smart_cols)
print('Fabricantes identificados:')
print(df['fabricante'].value_counts())


## 4. Análise Exploratória de Dados (EDA)

In [ ]:
df[smart_cols + ['capacity_tb']].describe().T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

qtd_fabricante = df.groupby('fabricante')['serial_number'].nunique().sort_values(ascending=False)
sns.barplot(x=qtd_fabricante.index, y=qtd_fabricante.values, ax=axes[0], palette='Blues_d')
axes[0].set_title('Discos únicos por fabricante')
axes[0].set_ylabel('Nº de discos')

top_modelos = df.groupby('model')['serial_number'].nunique().sort_values(ascending=False).head(10)
sns.barplot(y=top_modelos.index, x=top_modelos.values, ax=axes[1], palette='Greens_d')
axes[1].set_title('Top 10 modelos por quantidade de discos')
axes[1].set_xlabel('Nº de discos')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

capacidade_disco = df.drop_duplicates('serial_number')['capacity_tb']
sns.histplot(capacidade_disco, bins=20, kde=False, ax=axes[0], color='#3b6ea5')
axes[0].set_title('Distribuição de capacidade dos discos (TB)')
axes[0].set_xlabel('Capacidade (TB)')

registros_por_mes = df.set_index('date').resample('MS').size()
axes[1].plot(registros_por_mes.index, registros_por_mes.values, marker='o', color='#c0392b')
axes[1].set_title('Registros diários agregados por mês')
axes[1].set_ylabel('Nº de registros (disco-dia)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


**Leitura rápida da EDA:** a base está concentrada em poucos fabricantes (tipicamente Seagate, Western Digital, Toshiba e HGST) e a capacidade dos discos varia bastante — isso é relevante porque discos de maior capacidade reduzem a quantidade de unidades físicas necessárias para atingir a meta de armazenamento da nova unidade, impactando diretamente Custo e Tempo de instalação.

## 5. Análise de Risco

O risco é avaliado em **duas etapas complementares**:

**5.1 — Risco observado (dados reais):** a **Taxa de Falha Anualizada (AFR)** por modelo, calculada como:

$$
AFR_{modelo} = \frac{\text{falhas observadas}}{\text{disco-dias em operação}} \times 365 \times 100\%
$$

Modelos com poucos disco-dias acumulados são excluídos do ranking por gerarem estimativas estatisticamente instáveis.

**5.2 — Risco preditivo (Machine Learning):** um **Random Forest Classifier** treinado sobre os atributos S.M.A.R.T., para estimar a probabilidade de falha de um disco a partir de seus indicadores de saúde — permitindo identificar quais sinais mais antecedem uma falha.


In [ ]:
MIN_DISCO_DIAS = 5000  # limiar mínimo de disco-dias para o modelo entrar no ranking de risco

afr = (
    df.groupby('model')
      .agg(disco_dias=('serial_number', 'size'),
           falhas=('failure', 'sum'),
           fabricante=('fabricante', 'first'),
           capacidade_tb=('capacity_tb', 'first'))
)
afr = afr[afr['disco_dias'] >= MIN_DISCO_DIAS].copy()
afr['afr_pct'] = (afr['falhas'] / afr['disco_dias']) * 365 * 100
afr = afr.sort_values('afr_pct', ascending=False)

print(f'Modelos elegíveis para o ranking de risco (>= {MIN_DISCO_DIAS:,} disco-dias): {len(afr)}')
afr.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pior = afr.head(12).sort_values('afr_pct')
sns.barplot(y=pior.index, x=pior['afr_pct'], ax=axes[0], palette='Reds_r')
axes[0].set_title('Top 12 modelos — maior risco (AFR %)')
axes[0].set_xlabel('AFR (% ao ano)')

melhor = afr.tail(12).sort_values('afr_pct')
sns.barplot(y=melhor.index, x=melhor['afr_pct'], ax=axes[1], palette='Greens_r')
axes[1].set_title('Top 12 modelos — menor risco (AFR %)')
axes[1].set_xlabel('AFR (% ao ano)')

plt.tight_layout()
plt.show()


**Interpretação:** modelos com AFR muito acima da média do parque indicam maior probabilidade de substituição no primeiro ano de operação — o que eleva tanto o **custo de reposição** quanto o **tempo de resposta a incidentes** da nova unidade. Modelos com AFR baixo e capacidade alta tendem a ser os mais atrativos, desde que o custo por TB também seja competitivo (Seção 6).

In [ ]:
# Modelo preditivo de risco (Random Forest) a partir dos atributos S.M.A.R.T.
amostra = df.dropna(subset=smart_cols).copy()
if len(amostra) > 300000:
    amostra = amostra.sample(300000, random_state=RANDOM_STATE)  # amostragem para viabilizar o treino

X = amostra[smart_cols]
y = amostra['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

risk_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
risk_model.fit(X_train, y_train)

y_pred = risk_model.predict(X_test)
y_proba = risk_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

importances = pd.Series(risk_model.feature_importances_, index=smart_cols).sort_values()
importances.plot(kind='barh', ax=axes[0], color='#8e44ad')
axes[0].set_title('Importância dos atributos S.M.A.R.T. — modelo de risco')
axes[0].set_xlabel('Importância relativa')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#c0392b', label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('Taxa de falsos positivos')
axes[1].set_ylabel('Taxa de verdadeiros positivos')
axes[1].set_title('Curva ROC — modelo preditivo de falha')
axes[1].legend()

plt.tight_layout()
plt.show()


**Interpretação:** o modelo preditivo confirma, com dados reais, que atributos ligados a **setores realocados/pendentes e erros não corrigíveis** costumam ser os sinais mais fortes de falha iminente — um indicador operacional útil para a nova unidade priorizar a substituição preventiva de discos, reduzindo o risco de perda de dados antes que a falha completa ocorra.

## 6. Análise de Custo

A base do Backblaze **não contém preços** — por isso, o custo é modelado de forma heurística, combinando:

1. **Custo de aquisição**, a partir de premissas de mercado de US$/TB por fabricante (valores aproximados, apenas para fins didáticos do MVP).
2. **Custo esperado de reposição**, calculado como `AFR do modelo × custo unitário do disco × nº de discos necessários`, refletindo o impacto do risco real (Seção 5) sobre o custo total.

$$
Custo_{total} = \underbrace{n_{discos} \times custo\_TB \times capacidade\_TB}_{\text{aquisição}} \; + \; \underbrace{n_{discos} \times AFR\% \times custo\_unitário\_reposição}_{\text{reposição (1º ano)}}
$$


In [ ]:
# Premissas de mercado (US$/TB) — valores aproximados por fabricante, apenas para fins de MVP
CUSTO_USD_POR_TB = {
    'Seagate': 16.0,
    'Western Digital': 17.0,
    'Toshiba': 16.5,
    'HGST': 18.0,
    'Outro': 17.0,
}
CUSTO_LOGISTICO_REPOSICAO_USD = 60  # mão de obra + logística por disco substituído

META_CAPACIDADE_PB = 5  # capacidade-alvo da nova unidade, em petabytes
META_CAPACIDADE_TB = META_CAPACIDADE_PB * 1000

custo = afr.copy()
custo['custo_tb_usd'] = custo['fabricante'].map(CUSTO_USD_POR_TB)
custo['n_discos_necessarios'] = np.ceil(META_CAPACIDADE_TB / custo['capacidade_tb']).astype(int)
custo['custo_aquisicao_usd'] = custo['n_discos_necessarios'] * custo['custo_tb_usd'] * custo['capacidade_tb']
custo['custo_unit_disco_usd'] = custo['custo_tb_usd'] * custo['capacidade_tb']
custo['custo_reposicao_1ano_usd'] = (
    custo['n_discos_necessarios'] * (custo['afr_pct'] / 100) *
    (custo['custo_unit_disco_usd'] + CUSTO_LOGISTICO_REPOSICAO_USD)
)
custo['custo_total_1ano_usd'] = custo['custo_aquisicao_usd'] + custo['custo_reposicao_1ano_usd']

custo[['fabricante', 'capacidade_tb', 'n_discos_necessarios', 'custo_aquisicao_usd',
       'custo_reposicao_1ano_usd', 'custo_total_1ano_usd']].sort_values('custo_total_1ano_usd').head(10)


In [ ]:
top_custo = custo.sort_values('custo_total_1ano_usd').head(12)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_custo.index, top_custo['custo_aquisicao_usd'] / 1e6, color='#2c7fb8', label='Aquisição')
ax.barh(top_custo.index, top_custo['custo_reposicao_1ano_usd'] / 1e6,
        left=top_custo['custo_aquisicao_usd'] / 1e6, color='#e74c3c', label='Reposição (1º ano)')
ax.set_xlabel('Custo (US$ milhões)')
ax.set_title(f'Custo total estimado para {META_CAPACIDADE_PB} PB — Top 12 modelos mais econômicos')
ax.legend()
plt.tight_layout()
plt.show()


**Interpretação:** modelos com custo de aquisição um pouco maior podem ter custo total menor quando o risco de falha é baixo — evidenciando que **decidir só pelo preço do disco** pode ser um erro caro para uma unidade de armazenamento em nuvem, já que a reposição de discos tem custo real de logística e operação.

## 7. Estimativa de Tempo de Implantação

A base **não contém** uma variável de tempo de implantação — assim como no custo, trata-se de um **modelo heurístico de apoio à decisão**, com parâmetros explícitos (não derivados diretamente dos dados):

$$
T_{semanas} = \big(T_{aquisição}^{fabricante} + T_{instalação}(n_{discos})\big) \times \big(1 + 0{,}5 \cdot risco_{norm}\big)
$$

- **Tempo de aquisição** varia por fabricante (prazo de importação/logística de fornecimento).
- **Tempo de instalação** cresce com o número de discos/racks necessários.
- O **fator de risco** aumenta o prazo pela necessidade de testes de queima (*burn-in*) mais extensos em modelos historicamente menos confiáveis.


In [ ]:
TEMPO_AQUISICAO_SEMANAS = {
    'Seagate': 4,
    'Western Digital': 5,
    'Toshiba': 6,
    'HGST': 5,
    'Outro': 6,
}
DISCOS_POR_RACK = 60
SEMANAS_POR_RACK_INSTALACAO = 0.5  # inclui cabeamento, montagem e testes de queima iniciais

tempo = custo.copy()
tempo['n_racks'] = np.ceil(tempo['n_discos_necessarios'] / DISCOS_POR_RACK)
tempo['semanas_aquisicao'] = tempo['fabricante'].map(TEMPO_AQUISICAO_SEMANAS)
tempo['semanas_instalacao'] = tempo['n_racks'] * SEMANAS_POR_RACK_INSTALACAO

risco_norm_tmp = (tempo['afr_pct'] - tempo['afr_pct'].min()) / (tempo['afr_pct'].max() - tempo['afr_pct'].min())
tempo['tempo_estimado_semanas'] = (
    (tempo['semanas_aquisicao'] + tempo['semanas_instalacao']) * (1 + 0.5 * risco_norm_tmp)
).round(1)

tempo[['fabricante', 'n_discos_necessarios', 'n_racks', 'tempo_estimado_semanas']].sort_values('tempo_estimado_semanas').head(10)


In [ ]:
top_tempo = tempo.sort_values('tempo_estimado_semanas').head(12)

plt.figure(figsize=(9, 6))
sns.barplot(y=top_tempo.index, x=top_tempo['tempo_estimado_semanas'], palette='viridis')
plt.title(f'Tempo estimado de implantação — Top 12 modelos mais rápidos (meta: {META_CAPACIDADE_PB} PB)')
plt.xlabel('Semanas estimadas')
plt.tight_layout()
plt.show()


**Interpretação:** discos de maior capacidade reduzem o número de racks necessários e, portanto, o tempo de instalação — mas esse ganho pode ser anulado se o modelo tiver risco alto o suficiente para exigir testes de queima mais longos.

## 8. Matriz de Decisão Multicritério (MCDA)

Cada **modelo de disco elegível** é avaliado nas três dimensões — Risco, Custo e Tempo —, normalizadas em [0, 1] (min-max) e combinadas em um **score de decisão (0–100, maior é melhor)**:

$$
Score = 100 \times \Big(1 - \big(w_{risco} \cdot risco_{norm} + w_{custo} \cdot custo_{norm} + w_{tempo} \cdot tempo_{norm}\big)\Big)
$$


In [ ]:
def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

W_RISCO, W_CUSTO, W_TEMPO = 0.45, 0.35, 0.20  # pesos default (decisão gerencial)

mcda = tempo.copy()
mcda['risco_norm'] = normalize(mcda['afr_pct'])
mcda['custo_norm'] = normalize(mcda['custo_total_1ano_usd'])
mcda['tempo_norm'] = normalize(mcda['tempo_estimado_semanas'])
mcda['score_decisao'] = 100 * (1 - (W_RISCO * mcda['risco_norm'] + W_CUSTO * mcda['custo_norm'] + W_TEMPO * mcda['tempo_norm']))
mcda = mcda.sort_values('score_decisao', ascending=False)

top10 = mcda.head(10).copy()
plt.figure(figsize=(10, 6))
sns.barplot(y=top10.index, x=top10['score_decisao'], palette='mako')
plt.title('Top 10 modelos de disco — Score de decisão (Risco + Custo + Tempo)')
plt.xlabel('Score de decisão (0-100, maior é melhor)')
plt.tight_layout()
plt.show()

mcda[['fabricante', 'afr_pct', 'custo_total_1ano_usd', 'tempo_estimado_semanas', 'score_decisao']].head(10)


### 8.1 Análise de Sensibilidade dos Pesos

Como o score depende de pesos definidos pela gestão (e não estatisticamente derivados), verificamos **quão estável é o ranking do Top 5** diante de variações razoáveis nesses pesos, sorteando 300 combinações de pesos a partir de uma distribuição de Dirichlet centrada nos pesos default.


In [ ]:
np.random.seed(RANDOM_STATE)
n_sorteios = 300
top5_estabilidade = {i: 0 for i in mcda.index}

for _ in range(n_sorteios):
    w_r, w_c, w_t = np.random.dirichlet(alpha=[4.5, 3.5, 2.0])
    score_sim = 100 * (1 - (w_r * mcda['risco_norm'] + w_c * mcda['custo_norm'] + w_t * mcda['tempo_norm']))
    top5_idx = score_sim.sort_values(ascending=False).head(5).index
    for i in top5_idx:
        top5_estabilidade[i] += 1

mcda['estabilidade_top5_pct'] = pd.Series(top5_estabilidade) / n_sorteios * 100
mcda[['fabricante', 'score_decisao', 'estabilidade_top5_pct']].sort_values('score_decisao', ascending=False).head(5)


### 8.2 Simulação de Monte Carlo — Robustez sob Incerteza dos Dados

Além da incerteza nos pesos, existe incerteza amostral: o AFR de cada modelo é estimado a partir de uma amostra finita de disco-dias. A simulação abaixo faz **bootstrap** (reamostragem com reposição) dos registros diários de cada modelo para estimar a **probabilidade de cada modelo permanecer entre os 5 melhores** mesmo diante dessa incerteza.


In [ ]:
N_SIMULACOES = 200
top5_mc = {i: 0 for i in mcda.index}
modelos_top = mcda.head(15).index.tolist()  # restringe a simulação aos 15 melhores por custo computacional

pools = {m: df[df['model'] == m][['serial_number', 'failure']] for m in modelos_top}

for _ in range(N_SIMULACOES):
    linhas = []
    for m in modelos_top:
        pool = pools[m]
        amostra_boot = pool.sample(n=len(pool), replace=True, random_state=None)
        afr_boot = (amostra_boot['failure'].sum() / len(amostra_boot)) * 365 * 100
        row = mcda.loc[m].copy()
        row['afr_pct'] = afr_boot
        linhas.append(row)
    sim_df = pd.DataFrame(linhas, index=modelos_top)
    sim_df['risco_norm'] = normalize(sim_df['afr_pct'])
    sim_df['score_sim'] = 100 * (1 - (W_RISCO * sim_df['risco_norm'] + W_CUSTO * sim_df['custo_norm'] + W_TEMPO * sim_df['tempo_norm']))
    top5_idx = sim_df['score_sim'].sort_values(ascending=False).head(5).index
    for i in top5_idx:
        top5_mc[i] += 1

mcda['prob_top5_montecarlo_pct'] = pd.Series(top5_mc).reindex(mcda.index).fillna(0) / N_SIMULACOES * 100
mcda[['fabricante', 'score_decisao', 'estabilidade_top5_pct', 'prob_top5_montecarlo_pct']].sort_values('score_decisao', ascending=False).head(5)


## 9. Recomendação Final

In [ ]:
melhor = mcda.iloc[0]

print('=' * 70)
print('MODELO DE DISCO RECOMENDADO PARA A NOVA UNIDADE DE ARMAZENAMENTO')
print('=' * 70)
print(f"Modelo:                      {melhor.name}")
print(f"Fabricante:                  {melhor['fabricante']}")
print(f"Capacidade por disco:        {melhor['capacidade_tb']} TB")
print(f"Discos necessários ({META_CAPACIDADE_PB} PB):  {int(melhor['n_discos_necessarios']):,}")
print('-' * 70)
print(f"Score de decisão:            {melhor['score_decisao']:.1f} / 100")
print(f"AFR (risco):                 {melhor['afr_pct']:.2f}% ao ano")
print(f"Custo total estimado (1º ano): US$ {melhor['custo_total_1ano_usd']:,.0f}")
print(f"Tempo estimado de implantação: {melhor['tempo_estimado_semanas']:.1f} semanas")
print(f"Estabilidade no Top 5 (pesos): {melhor['estabilidade_top5_pct']:.0f}%")
print(f"Robustez no Top 5 (Monte Carlo): {melhor['prob_top5_montecarlo_pct']:.0f}%")
print('=' * 70)


In [ ]:
top5 = mcda.head(5).copy()
categorias = ['Risco (invertido)', 'Custo (invertido)', 'Tempo (invertido)']

fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
angles = np.linspace(0, 2 * np.pi, len(categorias), endpoint=False).tolist()
angles += angles[:1]

for idx, row in top5.iterrows():
    valores = [1 - row['risco_norm'], 1 - row['custo_norm'], 1 - row['tempo_norm']]
    valores += valores[:1]
    ax.plot(angles, valores, linewidth=2, label=str(idx))
    ax.fill(angles, valores, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categorias)
ax.set_yticklabels([])
ax.set_title('Top 5 modelos — comparação nas três dimensões (quanto maior, melhor)', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.show()


## 10. Exportação dos Resultados

In [ ]:
export_cols = ['fabricante', 'capacidade_tb', 'n_discos_necessarios', 'afr_pct',
               'custo_total_1ano_usd', 'tempo_estimado_semanas', 'score_decisao',
               'estabilidade_top5_pct', 'prob_top5_montecarlo_pct']
mcda[export_cols].to_csv('resultado_mcda_unidade_armazenamento.csv')

with open('relatorio_mvp_unidade_armazenamento.txt', 'w', encoding='utf-8') as f:
    f.write('MVP - Risco, Custo e Tempo de Implantacao de Nova Unidade de Armazenamento em Nuvem\n')
    f.write(f'Meta de capacidade: {META_CAPACIDADE_PB} PB\n\n')
    f.write(f'Modelo recomendado: {melhor.name} ({melhor["fabricante"]})\n')
    f.write(f'Score de decisao: {melhor["score_decisao"]:.1f}/100\n')
    f.write(f'AFR: {melhor["afr_pct"]:.2f}% ao ano\n')
    f.write(f'Custo total estimado (1o ano): US$ {melhor["custo_total_1ano_usd"]:,.0f}\n')
    f.write(f'Tempo estimado de implantacao: {melhor["tempo_estimado_semanas"]:.1f} semanas\n')

print('Arquivos exportados: resultado_mcda_unidade_armazenamento.csv, relatorio_mvp_unidade_armazenamento.txt')
files.download('resultado_mcda_unidade_armazenamento.csv')
files.download('relatorio_mvp_unidade_armazenamento.txt')


## 11. Conclusões e Limitações

**Principais achados:**
- O **risco** de falha varia fortemente entre modelos de disco — mesmo dentro do mesmo fabricante —, e atributos S.M.A.R.T. ligados a setores realocados/pendentes se confirmam, com dados reais, como fortes sinais de falha iminente.
- O **custo total** de propriedade não é dominado apenas pelo preço por TB: modelos mais baratos, mas com AFR alto, podem gerar custo de reposição relevante ao longo do primeiro ano.
- O **tempo de implantação** é reduzido por discos de maior capacidade (menos unidades físicas), mas pode ser parcialmente anulado pela necessidade de testes de queima mais longos em modelos de maior risco.
- A recomendação final busca equilibrar as três dimensões, e a análise de sensibilidade/Monte Carlo indica o quão sensível essa recomendação é a mudanças de prioridade da gestão e à incerteza amostral do risco.

**Limitações do MVP:**
- A base do Backblaze reflete o **parque de discos de uma única empresa**, sob suas condições específicas de operação (temperatura, carga de trabalho) — os resultados podem não generalizar para outros ambientes.
- Os parâmetros de **custo e tempo são premissas de mercado**, não valores contratuais reais; para uma decisão final, devem ser substituídos por cotações e SLAs de fornecedores.
- Modelos com poucos disco-dias acumulados foram excluídos do ranking de risco para evitar estimativas de AFR estatisticamente instáveis, o que reduz a base de modelos comparados.
